# Auditoria cruzada Athena × DuckDB
POSTECH – BB – Tech Challenge Fase 3

Comparação independente de 16 consultas. Scripts de origem: `sql/03_analytical_queries.sql` (blob fd41abc373af31096c78aa7abdce4345f9cf2cef) e `sql/04_complementary_queries.sql` (blob 8846fdd0ae55d111b37fa769e15117a3da18d078). Medianas calculadas com MEDIAN no DuckDB. Contagens e categorias exatas; decimais com tolerância absoluta 0,01 e relativa zero.

In [ ]:
%pip -q install duckdb

In [ ]:
import pandas as pd
import numpy as np
import duckdb, hashlib, zipfile
from io import BytesIO
from pathlib import PurePosixPath
from google.colab import files
from IPython.display import display
print('DuckDB:', duckdb.__version__, '| pandas:', pd.__version__)

## 1. Carregar o CSV original e o ZIP dos resultados Athena
Selecione os dois arquivos na mesma janela. Não envie o ZIP de evidências.

In [ ]:
uploaded = files.upload()
csvs = [k for k in uploaded if k.lower().endswith('.csv')]
zips = [k for k in uploaded if k.lower().endswith('.zip')]
assert len(csvs)==len(zips)==1 and len(uploaded)==2, 'Selecione exatamente um CSV e um ZIP.'
raw = uploaded[csvs[0]]
df = pd.read_csv(BytesIO(raw))
expected = 'customer_id credit_score country gender age tenure balance products_number credit_card active_member estimated_salary churn'.split()
assert list(df.columns)==expected
assert df.shape==(10000,12) and not df.isna().any().any()
assert df.customer_id.is_unique and not df.duplicated().any()
for c in ['churn','active_member','credit_card']:
    assert set(df[c].unique()) <= {0,1}
assert df.churn.sum()==2037
print('Dataset validado:', df.shape)
print('SHA-256 dataset:', hashlib.sha256(raw).hexdigest())
print('SHA-256 ZIP:', hashlib.sha256(uploaded[zips[0]]).hexdigest())

## 2. Inventário do ZIP
Ignora metadados do Finder; não extrai arquivos no sistema.

In [ ]:
rows = {'01':1,'02':1,'03':3,'04':2,'04_contingencia':2,'05':5,'05_complementar':5,'06a':2,'06b':6,'06c':2,'07':4,'08':2,'08_complementar':8,'09a':2,'09b':5,'10':2}
athena = {}
with zipfile.ZipFile(BytesIO(uploaded[zips[0]])) as z:
    for p in z.namelist():
        name = PurePosixPath(p).name
        if '__MACOSX' in PurePosixPath(p).parts or name.startswith('.') or not name.lower().endswith('.csv'):
            continue
        assert name.startswith('athena_q'), f'CSV inesperado: {name}'
        q = name[len('athena_q'):-4]
        assert q in rows and q not in athena, f'Arquivo inesperado ou duplicado: {name}'
        athena[q] = pd.read_csv(BytesIO(z.read(p)), dtype=str, keep_default_na=False)
assert set(athena)==set(rows), f'Faltantes: {set(rows)-set(athena)}'
display(pd.DataFrame([{'consulta':q,'linhas':len(athena[q]),'esperado':n} for q,n in rows.items()]))
print('Total esperado de linhas:', sum(rows.values()))

## 3. Consultas DuckDB
Versões SQL incorporadas para reprodutibilidade; MEDIAN substitui a expressão de posições centrais do Athena.

In [ ]:
queries = {
  "10": "SELECT\n    CASE\n        WHEN churn = 0 THEN 'Retidos'\n        ELSE 'Churn'\n    END AS situacao,\n    COUNT(*) AS total_clientes,\n    ROUND(AVG(estimated_salary), 2) AS salario_medio,\n    ROUND(MEDIAN(estimated_salary), 2) AS salario_mediano,\n    ROUND(STDDEV_SAMP(estimated_salary), 2) AS desvio_padrao\nFROM customers\nWHERE estimated_salary IS NOT NULL\nGROUP BY churn\nORDER BY churn",
  "01": "SELECT\n    COUNT(*) AS total_registros,\n    COUNT(DISTINCT customer_id) AS total_clientes\nFROM customers",
  "02": "SELECT\n    COUNT(*) AS total_clientes,\n    SUM(churn) AS clientes_churn,\n    COUNT(*) - SUM(churn) AS clientes_retidos,\n    ROUND(100.0 * AVG(churn), 2) AS taxa_churn_pct\nFROM customers",
  "03": "SELECT\n    country AS pais,\n    COUNT(*) AS total_clientes,\n    SUM(churn) AS clientes_churn,\n    COUNT(*) - SUM(churn) AS clientes_retidos,\n    ROUND(100.0 * AVG(churn), 2) AS taxa_churn_pct\nFROM customers\nGROUP BY country\nORDER BY taxa_churn_pct DESC",
  "04": "SELECT\n    gender AS genero,\n    COUNT(*) AS total_clientes,\n    SUM(churn) AS clientes_churn,\n    COUNT(*) - SUM(churn) AS clientes_retidos,\n    ROUND(100.0 * AVG(churn), 2) AS taxa_churn_pct\nFROM customers\nGROUP BY gender\nORDER BY taxa_churn_pct DESC",
  "05": "WITH clientes_faixa AS (\n    SELECT\n        customer_id,\n        churn,\n        CASE\n            WHEN age < 30 THEN 'Até 29 anos'\n            WHEN age < 40 THEN '30 a 39 anos'\n            WHEN age < 50 THEN '40 a 49 anos'\n            WHEN age < 60 THEN '50 a 59 anos'\n            ELSE '60 anos ou mais'\n        END AS faixa_etaria,\n        CASE\n            WHEN age < 30 THEN 1\n            WHEN age < 40 THEN 2\n            WHEN age < 50 THEN 3\n            WHEN age < 60 THEN 4\n            ELSE 5\n        END AS ordem\n    FROM customers\n)\n\nSELECT\n    faixa_etaria,\n    COUNT(*) AS total_clientes,\n    SUM(churn) AS clientes_churn,\n    COUNT(*) - SUM(churn) AS clientes_retidos,\n    ROUND(100.0 * AVG(churn), 2) AS taxa_churn_pct\nFROM clientes_faixa\nGROUP BY faixa_etaria, ordem\nORDER BY ordem",
  "06a": "SELECT\n    CASE\n        WHEN churn = 0 THEN 'Retidos'\n        ELSE 'Churn'\n    END AS situacao,\n    COUNT(*) AS total_clientes,\n    ROUND(AVG(balance), 2) AS saldo_medio,\n    ROUND(MEDIAN(balance), 2) AS saldo_mediano\nFROM customers\nWHERE balance IS NOT NULL\nGROUP BY churn\nORDER BY churn",
  "06b": "WITH clientes_saldo AS (\n    SELECT\n        churn,\n        CASE\n            WHEN balance = 0 THEN 'Saldo zero'\n            WHEN balance <= 50000 THEN 'Até 50 mil'\n            WHEN balance <= 100000 THEN '50 a 100 mil'\n            WHEN balance <= 150000 THEN '100 a 150 mil'\n            WHEN balance <= 200000 THEN '150 a 200 mil'\n            ELSE 'Acima de 200 mil'\n        END AS faixa_saldo,\n        CASE\n            WHEN balance = 0 THEN 1\n            WHEN balance <= 50000 THEN 2\n            WHEN balance <= 100000 THEN 3\n            WHEN balance <= 150000 THEN 4\n            WHEN balance <= 200000 THEN 5\n            ELSE 6\n        END AS ordem\n    FROM customers\n)\n\nSELECT\n    faixa_saldo,\n    COUNT(*) AS total_clientes,\n    SUM(churn) AS clientes_churn,\n    COUNT(*) - SUM(churn) AS clientes_retidos,\n    ROUND(100.0 * AVG(churn), 2) AS taxa_churn_pct\nFROM clientes_saldo\nGROUP BY faixa_saldo, ordem\nORDER BY ordem",
  "07": "SELECT\n    products_number AS quantidade_produtos,\n    COUNT(*) AS total_clientes,\n    SUM(churn) AS clientes_churn,\n    COUNT(*) - SUM(churn) AS clientes_retidos,\n    ROUND(100.0 * AVG(churn), 2) AS taxa_churn_pct\nFROM customers\nGROUP BY products_number\nORDER BY products_number",
  "08": "SELECT\n    CASE\n        WHEN active_member = 1 THEN 'Ativo'\n        ELSE 'Inativo'\n    END AS situacao_atividade,\n    COUNT(*) AS total_clientes,\n    SUM(churn) AS clientes_churn,\n    COUNT(*) - SUM(churn) AS clientes_retidos,\n    ROUND(100.0 * AVG(churn), 2) AS taxa_churn_pct\nFROM customers\nGROUP BY active_member\nORDER BY taxa_churn_pct DESC",
  "09b": "WITH clientes_score AS (\n    SELECT\n        churn,\n        CASE\n            WHEN credit_score < 500 THEN '350 a 499'\n            WHEN credit_score < 600 THEN '500 a 599'\n            WHEN credit_score < 700 THEN '600 a 699'\n            WHEN credit_score < 800 THEN '700 a 799'\n            ELSE '800 a 850'\n        END AS faixa_score\n    FROM customers\n)\n\nSELECT\n    faixa_score,\n    COUNT(*) AS total_clientes,\n    SUM(churn) AS clientes_churn,\n    COUNT(*) - SUM(churn) AS clientes_retidos,\n    ROUND(100.0 * AVG(churn), 2) AS taxa_churn_pct\nFROM clientes_score\nGROUP BY faixa_score\nORDER BY faixa_score",
  "04_contingencia": "SELECT\n    gender AS genero,\n    SUM(CASE WHEN churn = 0 THEN 1 ELSE 0 END) AS retidos,\n    SUM(CASE WHEN churn = 1 THEN 1 ELSE 0 END) AS churn\nFROM customers\nGROUP BY gender\nORDER BY gender",
  "05_complementar": "WITH clientes_faixa AS (\n    SELECT\n        churn,\n        CASE\n            WHEN age < 30 THEN 'Até 29 anos'\n            WHEN age < 40 THEN '30 a 39 anos'\n            WHEN age < 50 THEN '40 a 49 anos'\n            WHEN age < 60 THEN '50 a 59 anos'\n            ELSE '60 anos ou mais'\n        END AS faixa_etaria\n    FROM customers\n)\n\nSELECT\n    faixa_etaria,\n    COUNT(*) AS total_clientes,\n    SUM(churn) AS clientes_churn,\n    ROUND(100.0 * AVG(churn), 2) AS taxa_churn_pct,\n    ROUND(\n        100.0 * SUM(churn) /\n        (SELECT SUM(churn) FROM customers),\n        2\n    ) AS participacao_churn_pct\nFROM clientes_faixa\nGROUP BY faixa_etaria\nORDER BY clientes_churn DESC",
  "06c": "WITH clientes_saldo AS (\n    SELECT\n        churn,\n        CASE\n            WHEN balance = 0 THEN 'Saldo zero'\n            ELSE 'Saldo positivo'\n        END AS situacao_saldo\n    FROM customers\n)\n\nSELECT\n    situacao_saldo,\n    COUNT(*) AS total_clientes,\n    SUM(churn) AS clientes_churn,\n    ROUND(100.0 * AVG(churn), 2) AS taxa_churn_pct\nFROM clientes_saldo\nGROUP BY situacao_saldo\nORDER BY taxa_churn_pct",
  "08_complementar": "SELECT\n    products_number AS quantidade_produtos,\n    CASE\n        WHEN active_member = 1 THEN 'Ativo'\n        ELSE 'Inativo'\n    END AS situacao_atividade,\n    COUNT(*) AS total_clientes,\n    SUM(churn) AS clientes_churn,\n    ROUND(100.0 * AVG(churn), 2) AS taxa_churn_pct\nFROM customers\nGROUP BY products_number, active_member\nORDER BY products_number, active_member",
  "09a": "SELECT\n    CASE\n        WHEN churn = 0 THEN 'Retidos'\n        ELSE 'Churn'\n    END AS situacao,\n    COUNT(*) AS total_clientes,\n    ROUND(AVG(credit_score), 2) AS score_medio,\n    ROUND(MEDIAN(credit_score), 2) AS score_mediano,\n    ROUND(STDDEV_SAMP(credit_score), 2) AS desvio_padrao\nFROM customers\nWHERE credit_score IS NOT NULL\nGROUP BY churn\nORDER BY churn"
}
assert set(queries)==set(rows)
con = duckdb.connect(':memory:')
con.register('source_df', df)
con.execute('CREATE TABLE customers AS SELECT * FROM source_df')
duck = {q:con.sql(queries[q]).df() for q in rows}
print('Consultas executadas:',len(duck))

## 4. Comparação
Ordenação por chaves categóricas, comparação de colunas, dimensões, categorias e contagens. Não altera arquivos originais.

In [ ]:
keys = {'03':['pais'],'04':['genero'],'04_contingencia':['genero'],'05':['faixa_etaria'],'05_complementar':['faixa_etaria'],'06a':['situacao'],'06b':['faixa_saldo'],'06c':['situacao_saldo'],'07':['quantidade_produtos'],'08':['situacao_atividade'],'08_complementar':['quantidade_produtos','situacao_atividade'],'09a':['situacao'],'09b':['faixa_score'],'10':['situacao']}
exact = {'total_registros','total_clientes','clientes_churn','clientes_retidos','retidos','churn','quantidade_produtos'}
report, diffs = [], []
for q,n in rows.items():
    a,b = athena[q].copy(),duck[q].copy()
    issues = []
    try:
        assert list(a.columns)==list(b.columns), 'Colunas diferentes'
        assert len(a)==len(b)==n, 'Quantidade de linhas diferente'
        for c in b.columns:
            if c in exact or pd.api.types.is_numeric_dtype(b[c]):
                a[c] = pd.to_numeric(a[c], errors='raise')
                b[c] = pd.to_numeric(b[c], errors='raise')
                assert np.isfinite(a[c]).all() and np.isfinite(b[c]).all(), f'Valor não finito: {c}'
        k = keys.get(q,[])
        if k:
            assert not a.duplicated(k).any() and not b.duplicated(k).any(), 'Chaves duplicadas'
            a = a.sort_values(k)
            b = b.sort_values(k)
        a,b = a.reset_index(drop=True),b.reset_index(drop=True)
        for c in b.columns:
            if c in exact:
                ok = a[c].to_numpy()==b[c].to_numpy()
            elif pd.api.types.is_numeric_dtype(b[c]):
                ok = np.isclose(a[c],b[c],atol=0.010000001,rtol=0)
            else:
                ok = a[c].to_numpy()==b[c].to_numpy()
            for i in np.flatnonzero(~ok):
                diffs.append({'consulta':q,'linha':int(i)+1,'coluna':c,'athena':str(a.at[i,c]),'duckdb':str(b.at[i,c])})
                issues.append(f'{c} linha {i+1}')
    except (AssertionError, ValueError, TypeError) as e:
        issues.append(str(e))
    report.append({'consulta':q,'linhas_athena':len(a),'linhas_duckdb':len(b),'status':'DIVERGENTE' if issues else 'OK','detalhes':'; '.join(issues)})
relatorio = pd.DataFrame(report)
divergencias = pd.DataFrame(diffs,columns=['consulta','linha','coluna','athena','duckdb'])
display(relatorio)
display(divergencias)
total_ok = relatorio.status.eq('OK').sum()
print(f'Equivalência: {total_ok}/16 ({100*total_ok/16:.2f}%)')

## 5. Exportação
Baixe o relatório e o detalhamento, mesmo quando houver divergências.

In [ ]:
relatorio.to_csv('auditoria_athena_duckdb.csv',index=False)
divergencias.to_csv('divergencias_athena_duckdb.csv',index=False)
with zipfile.ZipFile('relatorio_auditoria.zip','w',compression=zipfile.ZIP_DEFLATED) as z:
    for f in ['auditoria_athena_duckdb.csv','divergencias_athena_duckdb.csv']:
        z.write(f)
files.download('relatorio_auditoria.zip')